In [1]:
#Q.13 상품 마스터 이상 진단과 지표 영향
# products.price는 object 컬럼 - 콤마 천단위('14,200'), '원' 접미사('12300원'), 0, 음수가 섞여 있음
# pd.to_numeric(errors="coerce")로 그대로 숫자화하면 형식 오염 값은 NaN -> 리포트가 NaN을 0으로 채우면(fillna(0)) 멀쩡한 상품도 마이너스 마진으로 잡힘
# '마스터 오류'인지 '진짜 역마진'인지는 마스터 price가 아니라 order_items 실거래 unit_price로 판정한다

In [ ]:
import pandas as pd

products = pd.read_csv("../data/products.csv", dtype={"price": str})
products = products.drop_duplicates(subset="product_id")  # product_id 3건 완전 중복 행 정리
products["category"] = products["category"].str.strip()

price_raw = products["price"]
is_missing = price_raw.isna() | (price_raw.str.strip() == "")
polluted = (~is_missing) & (price_raw.str.contains(",", na=False) | price_raw.str.contains("원", na=False))

price_num_raw = pd.to_numeric(price_raw, errors="coerce")  # 힌트 그대로: 형식 오염 -> NaN
price_clean = pd.to_numeric(
    price_raw.str.replace(",", "", regex=False).str.replace("원", "", regex=False), errors="coerce"
)  # 콤마·원 제거 후 재숫자화 (형식 오염의 '진짜 값' 추정치)

is_zero = (~is_missing) & (~polluted) & (price_clean == 0)
is_negative = (~is_missing) & (~polluted) & (price_clean < 0)
is_valid = (~is_missing) & (~polluted) & (price_clean > 0)
is_reverse_master = is_valid & (price_clean < products["cost"])  # 형식은 멀쩡한데 price < cost

issue_type = pd.Series("valid", index=products.index)
issue_type[is_missing] = "missing"
issue_type[polluted] = "format_polluted"
issue_type[is_zero] = "zero"
issue_type[is_negative] = "negative"
issue_type[is_reverse_master] = "reverse_margin_master"  # 유형은 겹치지 않게 배타적으로 부여 (뒤에 덮어쓴 조건이 우선)
products["issue_type"] = issue_type
products["price_raw_num"] = price_num_raw
products["price_clean"] = price_clean

products["issue_type"].value_counts().reindex(
    ["format_polluted", "zero", "negative", "missing", "reverse_margin_master", "valid"], fill_value=0
)

In [3]:
# order_items 실거래 단가로 이상 상품이 실제 얼마에 팔렸는지 대조
items = pd.read_csv("../data/order_items.csv", usecols=["order_id", "product_id", "unit_price"])
orders = pd.read_csv("../data/orders.csv", usecols=["order_id", "status"])
items = items.merge(orders, on="order_id", how="left")
items = items[items["status"] != "canceled"]  # 취소 주문은 실제 판매가 아니므로 제외

actual = (
    items.groupby("product_id")["unit_price"]
    .agg(sold_count="count", actual_price="mean", price_nunique="nunique")
    .reset_index()
)

anomalies = products[products["issue_type"] != "valid"].merge(actual, on="product_id", how="left")
anomalies["sold_count"] = anomalies["sold_count"].fillna(0).astype(int)

anomalies["master_margin"] = anomalies["price_raw_num"].fillna(0) - anomalies["cost"]  # 현재 리포트가 보는 마진(NaN->0 가정)
anomalies["actual_margin"] = anomalies["actual_price"] - anomalies["cost"]  # 실거래 단가 기준 진짜 마진
anomalies["margin_diff"] = anomalies["actual_margin"] - anomalies["master_margin"]
anomalies["impact_krw"] = anomalies["sold_count"] * anomalies["margin_diff"]  # 판매량 x 마진오차 = 정비 지연시 왜곡되는 금액

print(f"이상 상품 {len(anomalies)}건 중 마스터 기준 마이너스 마진: {(anomalies['master_margin'] < 0).sum()}건")
print(f"이상 상품 {len(anomalies)}건 중 실거래 기준 진짜 마이너스 마진: {(anomalies['actual_margin'] < 0).sum()}건")

anomalies[
    ["product_id", "product_name", "category", "issue_type", "price", "cost",
     "actual_price", "sold_count", "master_margin", "actual_margin", "margin_diff"]
].sort_values("issue_type")

이상 상품 54건 중 마스터 기준 마이너스 마진: 54건
이상 상품 54건 중 실거래 기준 진짜 마이너스 마진: 0건


,product_id,product_name,category,issue_type,price,cost,actual_price,sold_count,master_margin,actual_margin,margin_diff
26,362,프리미엄 청바지,의류,format_polluted,"34,000",14400.0,34000.0,356,-14400.0,19600.0,34000.0
52,333,스탠다드 코트,의류,format_polluted,"6,900",4700.0,6900.0,265,-4700.0,2200.0,6900.0
27,354,코어 에세이,도서,format_polluted,"22,000",14500.0,22000.0,1645,-14500.0,7500.0,22000.0
28,487,플러스 침대프레임,가구,format_polluted,45100원,30600.0,45100.0,261,-30600.0,14500.0,45100.0
29,473,코어 만화책,도서,format_polluted,"14,600",8800.0,14600.0,302,-8800.0,5800.0,14600.0
30,275,코어 잡지,도서,format_polluted,"19,900",13200.0,19900.0,951,-13200.0,6700.0,19900.0
31,350,베이직 의자,가구,format_polluted,"153,900",111600.0,153900.0,525,-111600.0,42300.0,153900.0
32,172,코어 충전기,전자,format_polluted,"83,100",55600.0,83100.0,291,-55600.0,27500.0,83100.0
33,84,코어 노트북,전자,format_polluted,"60,200",32500.0,60200.0,734,-32500.0,27700.0,60200.0
34,376,데일리 립스틱,뷰티,format_polluted,35400원,23300.0,35400.0,357,-23300.0,12100.0,35400.0


In [4]:
# 마스터 정비 우선순위 = 판매량 x 마진오차(impact_krw) 내림차순 - 고쳤을 때 리포트 숫자가 가장 많이 바뀌는 상품부터
priority = anomalies.sort_values("impact_krw", ascending=False)[
    ["product_id", "product_name", "issue_type", "price", "cost", "actual_price", "sold_count", "impact_krw"]
]
print(f"54건 정비 시 되찾는 총 마진 왜곡분: {anomalies['impact_krw'].sum():,.0f}원")
priority.head(10)

54건 정비 시 되찾는 총 마진 왜곡분: 2,460,579,500원


,product_id,product_name,issue_type,price,cost,actual_price,sold_count,impact_krw
39,92,데일리 시집,format_polluted,13500원,5800.0,13500.0,33752,455652000.0
46,53,프리미엄 키보드,zero,0,57900.0,129600.0,2116,274233600.0
15,195,프리미엄 책상,format_polluted,121900원,76400.0,121900.0,1210,147499000.0
2,252,스탠다드 이어폰,format_polluted,"95,500",46900.0,95500.0,1153,110111500.0
53,256,프리미엄 침대프레임,zero,0,40800.0,83100.0,1101,91493100.0
51,338,코어 잡지,format_polluted,16500원,7700.0,16500.0,5331,87961500.0
31,350,베이직 의자,format_polluted,"153,900",111600.0,153900.0,525,80797500.0
48,165,데일리 선반,format_polluted,"179,500",120200.0,179500.0,444,79698000.0
11,154,베이직 충전기,format_polluted,115800원,80100.0,115800.0,679,78628200.0
42,360,플러스 스탠드조명,zero,0,28200.0,53500.0,1415,75702500.0


### 제출물

**1) 이상 유형별 목록·건수** (products 497건 기준, 유형 간 배타적 분류)
- 형식 오염(콤마·'원' 접미사): **46건**
- 0원: **5건**
- 음수: **3건**
- 결측: **0건**
- 형식은 정상인데 price < cost(마스터 자체 역마진): **0건**
- 이상 합계: **54건**

**2) 마스터 vs 실거래 마진 차이표 (위 셀 2, 3)**
- 이상 54건을 order_items 실거래 unit_price와 대조한 결과, **actual_price는 형식 오염 46건 모두 콤마/원만 제거한 값과 정확히 일치**하고, 0원·음수 3건도 각각 정상 범위의 양수 가격으로 팔렸다.
- 반면 마스터 기준(NaN·0·음수를 그대로 반영) 마진은 **54건 전부 마이너스**로 잡힌다 — "마이너스 마진 상품이 무더기로 떴다"는 현상의 정체.
- 실거래 단가로 재계산하면 **actual_margin이 음수인 상품은 0건**. 즉 **진짜 역마진 상품은 하나도 없고, 54건 전부 상품 마스터 데이터 오류**다.

**3) 마스터 정비 우선순위 (위 셀 4, impact_krw = 판매량 × 마진오차 상위)**
1. 데일리 시집(id 92, 형식오염 '13500원') — 판매 33,752건, 왜곡 규모 약 4.6억원
2. 프리미엄 키보드(id 53, 0원) — 판매 2,116건, 왜곡 규모 약 2.7억원
3. 프리미엄 책상(id 195, 형식오염 '121900원') — 판매 1,210건, 왜곡 규모 약 1.5억원
4. 스탠다드 이어폰(id 252, 형식오염 '95,500') — 판매 1,153건, 왜곡 규모 약 1.1억원
5. 프리미엄 침대프레임(id 256, 0원) — 판매 1,101건, 왜곡 규모 약 0.9억원

54건 정비 시 되찾는 총 마진 왜곡분은 약 **24.6억원**. 형식 오염은 콤마·'원' 문자만 벗겨내면 즉시 해결되고, 0원·음수 8건은 값 자체를 실거래 평균단가로 교체하면 되므로 정비 난이도는 낮다 — **우선순위표 상위 판매량 순으로 일괄 수정**하는 것을 권고한다.